# CartoonStudio — Kaggle 3D story renderer

This notebook embeds the project source and produces an H.264 video with dialogue, lip-sync cues, character animation, lighting, and a story-selected scene. The bundled characters are procedural cartoon prototypes. For detailed cartoon or human output, supply rigged GLB/GLTF character assets with animation clips; the renderer can import those files directly.

Blender GPU support varies by Kaggle image and accelerator. Setup probes for a working Cycles CUDA device and uses EEVEE when Blender cannot use CUDA. Ara 3D Gen is an optional free character generator, but its full asset build can take a long time. The notebook reports whether imported characters actually include a skeleton, actions, and facial controls. This does not promise photorealistic output from the bundled starter assets.


In [ ]:
# Extract the project source embedded in this notebook (no Kaggle input dataset required).
from pathlib import Path
import base64, hashlib, zipfile, io, subprocess, sys

ROOT = Path('/kaggle/working/CartoonStudio')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE_B64 = '...'
archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
expected_sha256 = '965e2b5d6805b7ff020c60f5ecd0238a436055b943b36a2f97d224d7dec48fc7'
if hashlib.sha256(archive_bytes).hexdigest() != expected_sha256:
    raise RuntimeError('Embedded CartoonStudio source archive checksum mismatch.')
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    for name in z.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError(f'Unsafe path in embedded project archive: {name}')
    z.extractall('/kaggle/working')
print('Embedded source extracted:', ROOT)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'No NVIDIA GPU detected')


In [ ]:
# Install Blender, FFmpeg, Piper and a compatible Node runtime for Ara 3D Gen.
import subprocess, sys, shutil, os, platform, tarfile, io, requests
base_tools = ('ffmpeg', 'ffprobe', 'espeak', 'git', 'node', 'npm')
if not shutil.which('blender') or any(not shutil.which(tool) for tool in base_tools):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'blender', 'ffmpeg', 'espeak', 'git', 'nodejs', 'npm', 'xz-utils'], check=True)
if not shutil.which('blender'):
    raise RuntimeError('Blender installation failed: executable not found after apt install.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'piper-tts', 'requests'], check=True)
node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    # Kaggle images may ship Node 18. Install the current Node 22 LTS binary into
    # the session and put it first on PATH so npm uses it too.
    arch = {'x86_64':'x64', 'aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise RuntimeError(f'Unsupported Kaggle CPU architecture for Node: {platform.machine()}')
    releases = requests.get('https://nodejs.org/dist/index.json', timeout=30)
    releases.raise_for_status()
    release = next(item for item in releases.json() if item['version'].startswith('v22.') and f'linux-{arch}' in item['files'])
    version = release['version']
    archive_url = f'https://nodejs.org/dist/{version}/node-{version}-linux-{arch}.tar.xz'
    response = requests.get(archive_url, timeout=180)
    response.raise_for_status()
    node_root = Path('/kaggle/working/node22')
    node_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(response.content), mode='r:xz') as tf:
        base = f'node-{version}-linux-{arch}/'
        members = [m for m in tf.getmembers() if m.name.startswith(base)]
        for member in members:
            relative = Path(member.name[len(base):])
            target = (node_root / relative).resolve()
            if not target.is_relative_to(node_root.resolve()):
                raise RuntimeError('Unsafe path in Node.js release archive')
        tf.extractall(node_root, members=members, filter='data')
    os.environ['PATH'] = str(node_root/'bin') + os.pathsep + os.environ['PATH']
    node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    raise RuntimeError(f'Node 20+ is required by Ara 3D Gen; found {node_major}.')
print('Blender:', subprocess.check_output(['blender','--version'],text=True).splitlines()[0])
print('FFmpeg:', subprocess.check_output(['ffmpeg','-version'],text=True).splitlines()[0])
print('Node:', subprocess.check_output(['node','--version'],text=True).strip())


In [ ]:
# Select Cycles only when this Blender build can enumerate a CUDA device; Kaggle may expose an unusable CUDA runtime.
probe_expr="import bpy; a=bpy.context.preferences.addons.get('cycles'); p=a.preferences if a else None; p.compute_device_type='CUDA' if p else None; p.get_devices() if p else None; d=[x.name for x in p.devices if x.type=='CUDA'] if p else []; print('CYCLES_CUDA_DEVICES', d)"
RENDERER='eevee'; STRICT_GPU=False
try:
    probe=subprocess.run(['blender','-b','--python-expr',probe_expr],capture_output=True,text=True,timeout=35)
    print(probe.stdout[-1500:]); print(probe.stderr[-800:])
    if probe.returncode==0 and "CYCLES_CUDA_DEVICES ['" in probe.stdout:
        RENDERER='cycles'; STRICT_GPU=True
except subprocess.TimeoutExpired:
    print('Blender CUDA discovery timed out; using EEVEE.')
print('Selected renderer:',RENDERER,'strict GPU:',STRICT_GPU)


In [ ]:
# Prepare optional Ara generator, free phoneme lip-sync, and direct GLB/GLTF import support.
import json, os, requests, zipfile, time
from pathlib import Path
ARA_ROOT=Path('/kaggle/working/ara-3d-gen')
ARA_READY=False
try:
    if not (ARA_ROOT/'package.json').exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/olatunde85/3d-gen.git',str(ARA_ROOT)],check=True,timeout=300)
    if not (ARA_ROOT/'node_modules').exists():
        subprocess.run(['npm','ci'],cwd=ARA_ROOT,check=True,timeout=600)
    if not (ARA_ROOT/'public/assets/catalog.json').is_file():
        vendor=ARA_ROOT/'vendor'
        task='build-assets' if (vendor/'mh/makehuman/data').is_dir() and (vendor/'sys/skins').is_dir() else 'setup-assets'
        print('Ara asset task:',task,'(maximum 12 minutes)',flush=True)
        subprocess.run(['npm','run',task],cwd=ARA_ROOT,check=True,timeout=720)
    ARA_READY=(ARA_ROOT/'public/assets/catalog.json').is_file()
except (subprocess.TimeoutExpired,subprocess.CalledProcessError) as exc:
    print('Ara setup unavailable:',repr(exc),'The rest of the pipeline can use direct rigged GLB/GLTF assets or cartoon prototypes.',flush=True)
RHU_DIR=Path('/kaggle/working/rhubarb'); RHU_DIR.mkdir(parents=True,exist_ok=True)
rhubarb_binary=next(RHU_DIR.rglob('rhubarb'),None)
if rhubarb_binary is None:
    release=requests.get('https://api.github.com/repos/DanielSWolf/rhubarb-lip-sync/releases/latest',timeout=30); release.raise_for_status()
    asset=next((a for a in release.json()['assets'] if 'Linux.zip' in a['name']),None)
    if asset is None: raise RuntimeError('Rhubarb Linux release asset is unavailable.')
    archive=RHU_DIR/'rhubarb-linux.zip'; response=requests.get(asset['browser_download_url'],timeout=120); response.raise_for_status(); archive.write_bytes(response.content)
    with zipfile.ZipFile(archive) as zf: zf.extractall(RHU_DIR)
    rhubarb_binary=next(RHU_DIR.rglob('rhubarb'),None)
if rhubarb_binary is None: raise FileNotFoundError('Rhubarb binary missing after extraction.')
rhubarb_binary.chmod(0o755)
# For optional direct assets: create /kaggle/working/character_files.json, e.g. {"john":"/kaggle/input/my-assets/john.glb","sarah":"/kaggle/input/my-assets/sarah.glb"}.
CHARACTER_FILES=Path('/kaggle/working/character_files.json')
print('Ara catalog ready:',ARA_READY,'at',ARA_ROOT); print('Phoneme lip-sync ready:',rhubarb_binary)
print('Direct rigged assets configured:',CHARACTER_FILES.is_file())


In [ ]:
# Download one Piper voice directly from the public voice repository.
voices=ROOT/'voices'; voices.mkdir(exist_ok=True)
base='https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/alan/medium'
for filename in ('en_GB-alan-medium.onnx','en_GB-alan-medium.onnx.json'):
    path=voices/filename
    if not path.exists():
        response=requests.get(f'{base}/{filename}',timeout=180); response.raise_for_status(); path.write_bytes(response.content)
print('Voice model ready:',[(p.name,round(p.stat().st_size/1e6,1)) for p in voices.iterdir()])


In [ ]:
# Render the complete story at 720p/24; use a 3-second preview to check the pipeline first.
import time
OUT=Path('/kaggle/working/cartoon_studio_output'); OUT.mkdir(exist_ok=True)
cmd=[sys.executable,'-m','cartoon_studio.cli','generate','--story',str(ROOT/'stories'/'first_story.json'),'--voices',str(voices),'--work',str(OUT/'work'),'--output',str(OUT/'first_cartoon.mp4'),'--style','auto','--quality','high','--rhubarb',str(rhubarb_binary),'--tts','piper','--renderer',RENDERER,'--width','1280','--height','720','--fps','24','--samples','32','--preview-seconds','3']
if CHARACTER_FILES.is_file(): cmd += ['--character-files',str(CHARACTER_FILES)]
elif ARA_READY: cmd += ['--ara-root',str(ARA_ROOT)]
else: print('No detailed rigged character source configured. Output will use procedural prototypes; supply rigged assets for human-quality characters.')
if STRICT_GPU: cmd.append('--strict-gpu')
t0=time.monotonic(); subprocess.run(cmd,check=True,cwd=ROOT)
print('Video output:',OUT/'first_cartoon.mp4'); print('Total elapsed seconds:',round(time.monotonic()-t0,1))
from IPython.display import display,Video
display(Video(str(OUT/'first_cartoon.mp4'),embed=True))


In [ ]:
# Estimate render time from the measured frames when a render log is available.
import json
stats_path=OUT/'work'/'render_log.json'
if stats_path.exists():
    stats=json.loads(stats_path.read_text()); spf=stats['seconds_per_frame']
    subprocess.run([sys.executable,'-m','cartoon_studio.budget','--minutes','20','--seconds-per-frame',str(spf),'--weekly-hours','30','--reserve','0.15'],cwd=ROOT,check=True)
    print('Measured seconds per frame including Blender launch:',round(spf,3))
else: print('Render reused cached chunks; no fresh render timing was recorded.')


## Producing the full story

The render cell defaults to a short 3-second 1280×720 preview so you can confirm the configured assets before committing Kaggle GPU time. Once the result looks right, change `--preview-seconds 3` to `--preview-seconds 0` to render the full story. A ten-minute safety cap and resumable 20-second chunks are built into the CLI.

For detailed characters, attach freely licensed rigged `.glb` or `.gltf` files as Kaggle input and create `/kaggle/working/character_files.json` with one path per story character. Example: `{"john":"/kaggle/input/my-assets/john.glb","sarah":"/kaggle/input/my-assets/sarah.glb"}`. Those assets should include animation actions for body movement and facial blend shapes for expression and lip-sync. Cell 4 also attempts the optional Ara 3D Gen generator, but stops that setup after 12 minutes. It reports whether it is ready.

The default stage is a lightweight procedural set, not a scanned cinematic environment. If the notebook prints `procedural fallback`, that video is a pipeline preview and does not demonstrate realistic humans. For better results, pair detailed rigged character assets with a licensed environment/HDRI. The character report in Blender output lists skeleton, available action clips, and facial shape count so missing animation support is visible.
